In [1]:
!nvidia-smi
import torch, math
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Sat Nov 29 01:08:49 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch, math
import matplotlib.pyplot as plt

# Use GPU if available (fine to stay on CPU for L=1)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float64

# SU(2) constants
N = 2
C_V = 6.0 / N                    # from the Wilson Hessian bound
c0  = (N**2 - 1.0) / (2.0 * N)   # Haar mass coefficient from the cleaned proof

print("device:", device)
print("c0 =", c0, "C_V =", C_V)


device: cuda
c0 = 0.75 C_V = 3.0


In [3]:
# Build Pauli matrices on the chosen device
def build_paulis(device):
    sigma1 = torch.tensor([[0, 1],
                           [1, 0]], dtype=torch.complex128, device=device)
    sigma2 = torch.tensor([[0, -1j],
                           [1j, 0]], dtype=torch.complex128, device=device)
    sigma3 = torch.tensor([[1, 0],
                           [0,-1]], dtype=torch.complex128, device=device)
    return torch.stack([sigma1, sigma2, sigma3], dim=0)   # (3,2,2)

paulis = build_paulis(device)

def su2_from_vec(a_vec, paulis):
    """
    a_vec: (..., 3) real tensor (Lie algebra coordinates)
    paulis: (3,2,2) Pauli matrices
    returns: (..., 2, 2) complex SU(2) matrices via exp(i * a·σ/2)
    """
    a = a_vec.to(dtype=torch.float64, device=paulis.device)
    a_c = a.to(paulis.dtype)  # complex
    # A = (a_j sigma_j / 2)
    A = (a_c.unsqueeze(-1).unsqueeze(-1) * (paulis / 2.0)).sum(dim=-3)
    M = 1j * A  # i*A

    orig_shape = M.shape[:-2]
    M_flat = M.reshape(-1, 2, 2)
    U_flat = torch.zeros_like(M_flat)
    for idx in range(M_flat.shape[0]):
        U_flat[idx] = torch.matrix_exp(M_flat[idx])
    U = U_flat.reshape(*orig_shape, 2, 2)
    return U

# quick sanity check: U^\dagger U ~ I
test_vec = torch.randn(3, device=device)
U_test   = su2_from_vec(test_vec, paulis)
print("U^†U at test point:\n", (U_test.conj().transpose(-1,-2) @ U_test).real)


U^†U at test point:
 tensor([[1.0000e+00, 1.1053e-16],
        [1.1053e-16, 1.0000e+00]], device='cuda:0', dtype=torch.float64)


In [5]:
def make_links(A, a_lat, g, paulis):
    """
    A: (L,L,L,L,4,3) real Lie algebra field
    returns U: (L,L,L,L,4,2,2) complex link matrices
    """
    L = A.shape[0]
    A_flat = (a_lat * g * A).reshape(-1, 3)
    U_flat = su2_from_vec(A_flat, paulis)
    U = U_flat.reshape(L, L, L, L, 4, 2, 2)
    return U

def wilson_action(A, a_lat, g, beta, paulis):
    """
    Wilson action S_W = sum_p (1 - (1/N) Re Tr U_p), N=2.
    """
    L = A.shape[0]
    U = make_links(A, a_lat, g, paulis)
    S = torch.zeros((), dtype=dtype, device=A.device)

    for mu in range(4):
        for nu in range(mu+1, 4):
            for x0 in range(L):
                for x1 in range(L):
                    for x2 in range(L):
                        for x3 in range(L):
                            xs = [x0, x1, x2, x3]

                            # U_μ(x)
                            U1 = U[x0, x1, x2, x3, mu]

                            # U_ν(x+μ)
                            xp = xs.copy()
                            xp[mu] = (xp[mu] + 1) % L
                            U2 = U[xp[0], xp[1], xp[2], xp[3], nu]

                            # U_μ^†(x+ν)
                            xq = xs.copy()
                            xq[nu] = (xq[nu] + 1) % L
                            U3 = U[xq[0], xq[1], xq[2], xq[3], mu].conj().transpose(-1,-2)

                            # U_ν^†(x)
                            U4 = U[x0, x1, x2, x3, nu].conj().transpose(-1,-2)

                            U_p = U1 @ U2 @ U3 @ U4
                            tr  = torch.trace(U_p)
                            S   = S + (1.0 - 0.5 * tr.real)  # 1 - (1/2) ReTr
    return beta * S

def haar_action(A, a_lat, g, c0):
    """
    Quadratic Haar term: (c0/2) a^2 g^2 sum_b ||A_b||^2
    """
    return 0.5 * c0 * (a_lat**2) * (g**2) * (A**2).sum()

def effective_action(A, a_lat, g, beta, c0, paulis):
    return wilson_action(A, a_lat, g, beta, paulis) + haar_action(A, a_lat, g, c0)


In [6]:
from torch.autograd.functional import hessian

def hessian_at_point(L, a_lat, g, beta, c0, device='cpu', x0=None):
    """
    Compute Hessian of S_eff wrt all Lie-algebra coordinates on an L^4 lattice.
    x0: optional starting point (flattened), else uses zero field.
    Returns sorted eigenvalues as a 1D tensor.
    """
    paulis = build_paulis(device)
    n_params = L * L * L * L * 4 * 3

    def S_from_flat(flat):
        A = flat.view(L, L, L, L, 4, 3)
        return effective_action(A, a_lat, g, beta, c0, paulis)

    if x0 is None:
        x0 = torch.zeros(n_params, dtype=dtype, device=device, requires_grad=True)
    else:
        x0 = x0.clone().detach().to(dtype).to(device)
        x0.requires_grad_(True)

    H = hessian(S_from_flat, x0)
    evals, _ = torch.linalg.eigh(H.detach().cpu())
    return evals

def theoretical_rho_star(a_lat, g):
    beta = 2.0 * N / (g**2)
    return c0 * (a_lat**2) * (g**2) - beta * C_V


In [7]:
L     = 1
a_lat = 1.0
g     = 3.0
beta  = 2.0 * N / (g**2)

evals = hessian_at_point(L, a_lat, g, beta, c0, device=device)
rho   = theoretical_rho_star(a_lat, g)

print("Eigenvalues at A=0:", evals.numpy())
print("λ_min(H) =", evals[0].item())
print("ρ_*(a)    =", rho)


Eigenvalues at A=0: [6.75 6.75 6.75 6.75 6.75 6.75 6.75 6.75 6.75 6.75 6.75 6.75]
λ_min(H) = 6.750000000000001
ρ_*(a)    = 5.416666666666667


In [8]:
def scan_g_values(g_values, L=1, a_lat=1.0, device=device):
    results = []
    for g in g_values:
        beta = 2.0 * N / (g**2)
        rho_star = theoretical_rho_star(a_lat, g)
        evals = hessian_at_point(L, a_lat, g, beta, c0, device=device)
        lam_min = evals[0].item()
        results.append((g, rho_star, lam_min))
        print(f"g={g:.3f}  rho_*={rho_star:.4f}  lambda_min={lam_min:.4f}")
    return results

g_vals = [1.5, 2.0, 2.5, 3.0]
results = scan_g_values(g_vals, L=1, a_lat=1.0, device=device)


g=1.500  rho_*=-3.6458  lambda_min=1.6875
g=2.000  rho_*=0.0000  lambda_min=3.0000
g=2.500  rho_*=2.7675  lambda_min=4.6875
g=3.000  rho_*=5.4167  lambda_min=6.7500


In [1]:
import numpy as np
from numpy.linalg import svd

# ============================================================
# 1. Gauss–Legendre tensor for 2D U(1) gauge theory with θ-term
#    S = -β ∑_x cos p_x - i θ Q
#    p_x = φ_x,1 + φ_{x+1,2} - φ_{x+2,1} - φ_{x,2}
#    q_x = p_x mod 2π ∈ [-π,π], Q = (1/2π) ∑_x q_x
#
# Local tensor (continuous):
#   T(φ1,φ2,φ3,φ4) = exp[ β cos p + i (θ/2π) q ]
#
# Discretization by Gauss–Legendre on [-π,π]:
#   φ_i = π x_i,  w_φ,i = π w_i
#   T_ijkl = sqrt(w_i w_j w_k w_l)/(2π)^2 * T(φ_i,φ_j,φ_k,φ_l)
# ============================================================

def make_u1_theta_tensor(beta, theta, K):
    """
    Build the rank-4 local tensor for 2D U(1) LGT with θ term
    using K-point Gauss–Legendre quadrature on [-π, π].

    Indices: T[r,u,l,d] with angles:
        φ_r = φ_nodes[r], φ_u = φ_nodes[u],
        φ_l = φ_nodes[l], φ_d = φ_nodes[d]
    and
        p = φ_r + φ_u - φ_l - φ_d
        q = (p mod 2π) in [-π, π]
    """

    # Gauss–Legendre nodes/weights on [-1,1]
    x, w = np.polynomial.legendre.leggauss(K)
    # Map to [-π,π]: φ = π x, dφ = π dx
    phi_nodes = np.pi * x
    w_phi = np.pi * w  # weights for ∫_{-π}^{π} f(φ) dφ

    T = np.empty((K, K, K, K), dtype=np.complex128)

    two_pi = 2.0 * np.pi
    denom_norm = (two_pi ** 2)

    for r in range(K):
        φr = phi_nodes[r]
        wr = w_phi[r]
        for u in range(K):
            φu = phi_nodes[u]
            wu = w_phi[u]
            for l in range(K):
                φl = phi_nodes[l]
                wl = w_phi[l]
                for d in range(K):
                    φd = phi_nodes[d]
                    wd = w_phi[d]

                    # plaquette angle
                    p = φr + φu - φl - φd

                    # wrap to [-π, π] (field-theoretic definition of q_x)
                    q = (p + np.pi) % (two_pi) - np.pi

                    # local Boltzmann weight
                    weight = np.exp(beta * np.cos(p) + 1j * theta * q / two_pi)

                    # quadrature prefactor for 4 link integrals
                    pref = np.sqrt(wr * wu * wl * wd) / denom_norm

                    T[r, u, l, d] = pref * weight

    return T, phi_nodes, w_phi


# ============================================================
# 2. Levin–Nave TRG on a square lattice (Cook's algorithm)
#
# T has indices [r,u,l,d] (right, up, left, down).
#
# Each iteration:
#   - build Ma, Mb (D^2 × D^2),
#   - SVD → four 3-index tensors S1..S4,
#   - decimate an extraneous square → new T' (rank-4, D_new^4),
#   - lattice size L → L/2 in each direction (L → L/2).
#
# After 'no_iter' steps, L = 2^no_iter. Z is sum of the final T.
# ============================================================

def trg_step(T, Dcut):
    """
    One TRG step on a rank-4 tensor T[r,u,l,d] with bond dimension D.

    Returns:
        T_new: coarse-grained tensor
    """
    D = T.shape[0]
    assert T.shape == (D, D, D, D)

    inds = np.arange(D)

    # D_new = min(D^2, Dcut)
    D_new = min(D * D, Dcut)
    inds_new = np.arange(D_new)

    # --- build Ma and Mb as in Cook's TRG code ---

    Ma = np.empty((D * D, D * D), dtype=np.complex128)
    Mb = np.empty((D * D, D * D), dtype=np.complex128)

    # Ma[l + D*u, r + D*d] = T[r,u,l,d]
    # Mb[l + D*d, r + D*u] = T[r,u,l,d]
    for r in inds:
        for u in inds:
            for l in inds:
                for d in inds:
                    val = T[r, u, l, d]
                    Ma[l + D * u, r + D * d] = val
                    Mb[l + D * d, r + D * u] = val

    # --- SVD on Ma: produce S1, S3 ---

    U, s, Vh = svd(Ma, full_matrices=False)

    # keep largest D_new singular values
    idx = np.argsort(s)[::-1][:D_new]
    s_cut = s[idx]
    U_cut = U[:, idx]      # shape (D^2, D_new)
    Vh_cut = Vh[idx, :]    # shape (D_new, D^2)

    S1 = np.empty((D, D, D_new), dtype=np.complex128)
    S3 = np.empty((D, D, D_new), dtype=np.complex128)

    for x in inds:
        for y in inds:
            xy = x + D * y
            for m in inds_new:
                sqrt_s = np.sqrt(s_cut[m])
                S1[x, y, m] = sqrt_s * U_cut[xy, m]
                S3[x, y, m] = sqrt_s * Vh_cut[m, xy]

    # --- SVD on Mb: produce S2, S4 ---

    U, s, Vh = svd(Mb, full_matrices=False)
    idx = np.argsort(s)[::-1][:D_new]
    s_cut = s[idx]
    U_cut = U[:, idx]
    Vh_cut = Vh[idx, :]

    S2 = np.empty((D, D, D_new), dtype=np.complex128)
    S4 = np.empty((D, D, D_new), dtype=np.complex128)

    for x in inds:
        for y in inds:
            xy = x + D * y
            for m in inds_new:
                sqrt_s = np.sqrt(s_cut[m])
                S2[x, y, m] = sqrt_s * U_cut[xy, m]
                S4[x, y, m] = sqrt_s * Vh_cut[m, xy]

    # --- decimate extraneous squares → T_new ---

    T_new = np.zeros((D_new, D_new, D_new, D_new), dtype=np.complex128)

    for r in inds_new:
        for u in inds_new:
            for l in inds_new:
                for d in inds_new:
                    s_acc = 0.0 + 0.0j
                    for a in inds:
                        for b in inds:
                            for g in inds:
                                for w in inds:
                                    s_acc += (S1[w, a, r] *
                                              S2[a, b, u] *
                                              S3[b, g, l] *
                                              S4[g, w, d])
                    T_new[r, u, l, d] = s_acc

    return T_new


def Z_TRG_u1(beta, theta, K, Dcut, no_iter):
    """
    Compute Z(β, θ) on an L×L lattice with L = 2^no_iter
    for 2D U(1) gauge theory with θ-term using TRG.

    Args:
        beta: gauge coupling β
        theta: topological angle θ
        K: Gauss–Legendre points (bond dimension of initial tensor)
        Dcut: TRG truncation dimension
        no_iter: number of TRG iterations (L = 2^no_iter)

    Returns:
        Z (complex): TRG-approximated partition function
        L (int): linear lattice size
    """
    T, _, _ = make_u1_theta_tensor(beta, theta, K)

    D = K
    for n in range(no_iter):
        T = trg_step(T, Dcut)
        D = T.shape[0]

    Z = np.sum(T)
    L = 2 ** no_iter
    return Z, L


# ============================================================
# 3. Free energy and χ_top extraction
# ============================================================

def free_energy_density(beta, theta, K=8, Dcut=8, no_iter=4):
    """
    F(θ) per site ≈ -(1/L^2) log |Z|.
    """
    Z, L = Z_TRG_u1(beta, theta, K, Dcut, no_iter)
    V = L * L
    F = - np.log(np.abs(Z)) / V
    return F.real  # imaginary part is from arg(Z), we ignore it in F


def chi_top_finite_difference(beta, h, K=8, Dcut=8, no_iter=4):
    """
    Crude estimator of χ_top = F''(0) using symmetric finite differences.

        χ_top ≈ (F(+h) - 2 F(0) + F(-h)) / h^2
    """
    F0  = free_energy_density(beta, 0.0,   K=K, Dcut=Dcut, no_iter=no_iter)
    Fp  = free_energy_density(beta, +h,    K=K, Dcut=Dcut, no_iter=no_iter)
    Fm  = free_energy_density(beta, -h,    K=K, Dcut=Dcut, no_iter=no_iter)
    return (Fp - 2.0 * F0 + Fm) / (h * h)


# ============================================================
# 4. Example usage / quick sanity checks
# ============================================================

if __name__ == "__main__":
    # Parameters you can tweak:
    beta   = 0.0       # strong-coupling limit
    K      = 4         # Gauss–Legendre points (start small; K=8–16 later)
    Dcut   = 4         # TRG bond truncation
    no_iter = 2        # L = 2^no_iter

    # Quick check: at β=0 and θ=0, Z should be ~1 (up to TRG/quadrature errors)
    Z0, L = Z_TRG_u1(beta, 0.0, K, Dcut, no_iter)
    print(f"beta={beta}, theta=0.0 → Z ≈ {Z0}, L={L}, |Z|={abs(Z0):.8f}")

    # Scan a small θ-grid and compute F(θ)
    thetas = np.linspace(0.0, 2.0*np.pi, 9)
    Fs = []
    for th in thetas:
        Fth = free_energy_density(beta, th, K=K, Dcut=Dcut, no_iter=no_iter)
        Fs.append(Fth)
        print(f"theta={th:.3f}, F(θ)≈{Fth:.8e}")

    # Crude χ_top from finite difference around θ=0
    chi_est = chi_top_finite_difference(beta, h=0.1, K=K, Dcut=Dcut, no_iter=no_iter)
    print(f"Estimated χ_top(β={beta}) ≈ {chi_est:.8e}")


beta=0.0, theta=0.0 → Z ≈ (0.9999999999999978+0j), L=4, |Z|=1.00000000
theta=0.000, F(θ)≈1.38777878e-16
theta=0.785, F(θ)≈5.93946888e-03
theta=1.571, F(θ)≈2.30652250e-02
theta=2.356, F(θ)≈4.70134872e-02
theta=3.142, F(θ)≈1.00771607e-01
theta=3.927, F(θ)≈4.67291382e-02
theta=4.712, F(θ)≈2.30618670e-02
theta=5.498, F(θ)≈5.94350712e-03
theta=6.283, F(θ)≈1.66533454e-16
Estimated χ_top(β=0.0) ≈ 1.79425444e-02


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
from functools import partial

jax.config.update("jax_enable_x64", True)

# ----------------------------
# 1. Gauss–Legendre on [-π, π]
# ----------------------------

def gauss_legendre_nodes_weights(K: int):
    x, w = np.polynomial.legendre.leggauss(K)  # host-side once
    phi = np.pi * x
    wphi = np.pi * w
    return jnp.asarray(phi), jnp.asarray(wphi)

# ----------------------------
# 2. Local U(1)+θ tensor
# ----------------------------

@jax.jit
def make_u1_theta_tensor(beta, theta, phi_nodes, w_phi):
    """
    T[r,u,l,d] ∝ exp[ β cos p + i (θ/2π) q ],
      p = φ_r + φ_u − φ_l − φ_d,
      q = (p mod 2π) ∈ [−π, π].
    """
    beta = jnp.asarray(beta)
    theta = jnp.asarray(theta)

    phi_r, phi_u, phi_l, phi_d = jnp.meshgrid(
        phi_nodes, phi_nodes, phi_nodes, phi_nodes, indexing="ij"
    )
    w_r, w_u, w_l, w_d = jnp.meshgrid(
        w_phi, w_phi, w_phi, w_phi, indexing="ij"
    )

    two_pi = 2.0 * jnp.pi
    p = phi_r + phi_u - phi_l - phi_d
    q = (p + jnp.pi) % two_pi - jnp.pi

    weight = jnp.exp(beta * jnp.cos(p) + 1j * theta * q / two_pi)
    pref = jnp.sqrt(w_r * w_u * w_l * w_d) / (two_pi ** 2)
    return pref * weight  # (K,K,K,K), complex128

# ----------------------------
# 3. Single TRG step (Levin–Nave)
# ----------------------------

@partial(jax.jit, static_argnums=(1,))
def trg_step(T, Dcut: int):
    """
    One TRG step for T[r,u,l,d]; bond dimension truncated to Dcut.
    """
    D = T.shape[0]
    assert T.shape == (D, D, D, D)
    assert Dcut <= D * D
    D_new = Dcut

    # Ma[(l,u),(r,d)] = T[r,u,l,d]
    T_lurd = jnp.transpose(T, (2, 1, 0, 3))
    Ma = jnp.reshape(T_lurd, (D * D, D * D))

    # Mb[(l,d),(r,u)] = T[r,u,l,d]
    T_ldru = jnp.transpose(T, (2, 3, 0, 1))
    Mb = jnp.reshape(T_ldru, (D * D, D * D))

    # --- SVD Ma -> S1, S3 ---
    Ua, sa, Vha = jnp.linalg.svd(Ma, full_matrices=False)
    Ua = Ua[:, :D_new]
    sa = sa[:D_new]
    Vha = Vha[:D_new, :]

    sqrt_sa = jnp.sqrt(sa)
    sqrt_sa_b = sqrt_sa[None, None, :]

    Ua_rs = jnp.reshape(Ua, (D, D, D_new))          # (w,a,r)
    Vha_rs = jnp.reshape(Vha, (D_new, D, D))        # (m,b,g)

    S1 = Ua_rs * sqrt_sa_b                          # S1[w,a,r]
    S3 = jnp.transpose(Vha_rs, (1, 2, 0)) * sqrt_sa_b  # S3[b,g,l]

    # --- SVD Mb -> S2, S4 ---
    Ub, sb, Vhb = jnp.linalg.svd(Mb, full_matrices=False)
    Ub = Ub[:, :D_new]
    sb = sb[:D_new]
    Vhb = Vhb[:D_new, :]

    sqrt_sb = jnp.sqrt(sb)
    sqrt_sb_b = sqrt_sb[None, None, :]

    Ub_rs = jnp.reshape(Ub, (D, D, D_new))
    Vhb_rs = jnp.reshape(Vhb, (D_new, D, D))

    S2 = Ub_rs * sqrt_sb_b                          # S2[a,b,u]
    S4 = jnp.transpose(Vhb_rs, (1, 2, 0)) * sqrt_sb_b  # S4[g,w,d]

    # --- Contract S1..S4 → T_new ---
    # T_new[r,u,l,d] = Σ_{w,a,b,g} S1[w,a,r] S2[a,b,u] S3[b,g,l] S4[g,w,d]
    T_new = jnp.einsum("war,abu,bgl,gwd->ruld", S1, S2, S3, S4)
    return T_new

# ----------------------------
# 4. Z(β, θ) via TRG loop
# ----------------------------

def Z_TRG_u1(beta, theta, phi_nodes, w_phi, Dcut: int, no_iter: int):
    T0 = make_u1_theta_tensor(beta, theta, phi_nodes, w_phi)

    def body_fun(_, T):
        return trg_step(T, Dcut)

    T_final = jax.lax.fori_loop(0, no_iter, body_fun, T0)
    Z = jnp.sum(T_final)
    L = 2 ** no_iter
    return Z, L

Z_TRG_u1_jit = jax.jit(Z_TRG_u1, static_argnums=(4, 5))

# ----------------------------
# 5. F(θ) and χ_top
# ----------------------------

def free_energy_density(beta, theta, phi_nodes, w_phi, Dcut: int, no_iter: int):
    Z, L = Z_TRG_u1_jit(beta, theta, phi_nodes, w_phi, Dcut, no_iter)
    V = L * L
    F = -jnp.log(jnp.abs(Z)) / V
    return jnp.real(F)

def chi_top_finite_difference(beta, h, phi_nodes, w_phi, Dcut: int, no_iter: int):
    F0 = free_energy_density(beta, 0.0, phi_nodes, w_phi, Dcut, no_iter)
    Fp = free_energy_density(beta, +h,  phi_nodes, w_phi, Dcut, no_iter)
    Fm = free_energy_density(beta, -h,  phi_nodes, w_phi, Dcut, no_iter)
    return (Fp - 2.0 * F0 + Fm) / (h * h)

def free_energy_theta_grid(beta, thetas, phi_nodes, w_phi, Dcut: int, no_iter: int):
    thetas = jnp.asarray(thetas)
    F_single = lambda th: free_energy_density(beta, th, phi_nodes, w_phi, Dcut, no_iter)
    return jax.vmap(F_single)(thetas)

# ----------------------------
# 6. Example: use the T4
# ----------------------------

if __name__ == "__main__":
    # You can crank these up; start moderate to not explode SVD flops.
    K      = 16      # Gauss–Legendre points (local bond dim)
    Dcut   = 16      # TRG truncation (≤ K^2)
    no_iter = 4      # L = 2^no_iter

    beta = 0.0
    h    = 0.05

    phi_nodes, w_phi = gauss_legendre_nodes_weights(K)

    # Warm-up / compile
    Z0, L = Z_TRG_u1_jit(beta, 0.0, phi_nodes, w_phi, Dcut, no_iter)
    print(f"beta={beta}, theta=0.0 → L={L}, |Z|={float(jnp.abs(Z0)):.8f}")

    # θ-grid scan (vmapped)
    thetas = jnp.linspace(0.0, 2.0 * jnp.pi, 17)
    F_vals = free_energy_theta_grid(beta, thetas, phi_nodes, w_phi, Dcut, no_iter)
    for th, Fth in zip(np.array(thetas), np.array(F_vals)):
        print(f"theta={th:6.3f}, F(θ)≈{Fth:.8e}")

    # χ_top estimate at θ=0
    chi_est = chi_top_finite_difference(beta, h, phi_nodes, w_phi, Dcut, no_iter)
    print(f"Estimated χ_top(β={beta}) ≈ {float(chi_est):.8e}")
